## Day 14 — Final Production-Ready System

**The capstone day. Two tasks:**
1. **Task 1** — Run the full combined pipeline: Bronze → Silver → Gold → ML → Inference
2. **Task 2** — Save and confirm the final model in MLflow registry

**Plus production hardening:**
- Data quality assertions after each stage
- Model health check (AUC + Recall)
- Drift monitor (purchase rate vs baseline)

**This notebook is the production-ready version of the entire 14-day challenge.**

In [0]:
import time
import mlflow
from mlflow.tracking import MlflowClient
from pyspark.sql.functions import (
    col, count, avg, round as spark_round,
    when, isnan, isnull, min as spark_min, max as spark_max
)

# ── CONFIGURATION ─────────────────────────────────────────────────
CATALOG      = 'ecommerce'
BRONZE_TBL   = f'{CATALOG}.bronze.events_raw'
SILVER_TBL   = f'{CATALOG}.silver.events_si'
GOLD_FEAT    = f'{CATALOG}.gold.user_features'
GOLD_PRED    = f'{CATALOG}.gold.user_predictions'
MODEL_NAME   = f'{CATALOG}.gold.purchase_model'
MODEL_ALIAS  = 'rf_best_tuned_v1'

# Production thresholds
MIN_AUC           = 0.92
MIN_RECALL        = 0.90
BASELINE_PURCHASE = 1.51   # % from Day 5
DRIFT_THRESHOLD   = 15.0   # % relative change allowed

pipeline_log = []   # collect all stage results for final summary
print('Configuration ready')
print(f'  Catalog:      {CATALOG}')
print(f'  Model:        {MODEL_NAME}')
print(f'  Min AUC:      {MIN_AUC}')
print(f'  Drift limit:  {DRIFT_THRESHOLD}%')


### Task 1 — Combined Pipeline

**Stage 1: Pipeline health check** — verify all tables exist before running.

In [0]:
# Verify every table in the pipeline exists and has data
print('=== PIPELINE HEALTH CHECK ===')
print()

tables = [
    (BRONZE_TBL,  'Bronze', 'Raw events'),
    (SILVER_TBL,  'Silver', 'Cleaned events'),
    (GOLD_FEAT,   'Gold',   'User features'),
    (GOLD_PRED,   'Gold',   'User predictions'),
]

all_healthy = True
print(f'{"Table":<45} {"Layer":<10} {"Status":<10} {"Rows":>15}')
print('-' * 85)
for tbl, layer, desc in tables:
    try:
        n = spark.table(tbl).count()
        status = 'OK' if n > 0 else 'EMPTY'
        if n == 0: all_healthy = False
        print(f'{tbl:<45} {layer:<10} {status:<10} {n:>15,}')
        pipeline_log.append({'stage': desc, 'rows': n, 'status': status})
    except Exception as e:
        print(f'{tbl:<45} {layer:<10} {"MISSING":<10} {0:>15,}  <- {e}')
        all_healthy = False

print()
print(f'Pipeline health: {"ALL TABLES OK" if all_healthy else "WARNING — CHECK ABOVE"}')


**Stage 2: Bronze validation** — schema and row count check.

In [0]:
# Bronze check — verify raw data is intact
print('=== BRONZE LAYER VALIDATION ===')
t0 = time.time()

bronze = spark.table(BRONZE_TBL)
n_bronze = bronze.count()

# Schema check
expected_cols = {'user_id', 'product_id', 'event_type', 'event_time', 'price'}
actual_cols   = set(bronze.columns)
missing_cols  = expected_cols - actual_cols

# Event type distribution
event_dist = bronze.groupBy('event_type').count().orderBy('count')

print(f'Row count       : {n_bronze:,}')
print(f'Schema check    : {"PASS" if not missing_cols else f"FAIL — missing: {missing_cols}"}')
print(f'Columns present : {sorted(actual_cols)}')
print(f'Time            : {round(time.time()-t0,2)}s')
print()
print('Event distribution:')
event_dist.show()

# Production assertion
assert n_bronze > 100_000_000, f'Bronze row count too low: {n_bronze:,} — expected 109M+'
print('Bronze assertion PASSED')


**Stage 3: Silver quality check** — null rates, partition confirmation.

In [0]:
# Silver quality check — nulls, schema, partitions
print('=== SILVER LAYER QUALITY CHECK ===')
t0 = time.time()

silver = spark.table(SILVER_TBL)
n_silver = silver.count()

# Null check on key columns
key_cols = ['user_id', 'product_id', 'event_type', 'price']
print(f'Row count: {n_silver:,}')
print()
print('Null rates on key columns:')
null_issues = False
for c in key_cols:
    if c in silver.columns:
        n_null = silver.filter(col(c).isNull()).count()
        rate   = round(n_null / n_silver * 100, 3)
        status = 'OK' if rate == 0 else 'WARNING'
        if rate > 0: null_issues = True
        print(f'  {c:<20}: {n_null:>8,} nulls  ({rate}%)  {status}')

# Partition info
print()
spark.sql(f'DESCRIBE DETAIL {SILVER_TBL}').select('partitionColumns','numFiles','sizeInBytes').show(truncate=False)

print(f'Time: {round(time.time()-t0,2)}s')
print(f'Silver quality: {"WARNING — null values found" if null_issues else "PASS — no nulls on key columns"}')


**Stage 4: Gold features check** — shape, feature ranges, no negative values.

In [0]:
# Gold features quality check
print('=== GOLD FEATURES CHECK ===')
t0 = time.time()

feat = spark.table(GOLD_FEAT)
n_users = feat.count()

print(f'User count : {n_users:,}')
print(f'Features   : {feat.columns}')
print()

# Feature range check — all counts should be >= 0
feat_cols = ['total_events', 'total_sessions', 'total_views', 'total_cart_adds']
print('Feature ranges (sanity check):')
for fc in feat_cols:
    if fc in feat.columns:
        stats = feat.select(spark_min(col(fc)).alias('min'), spark_max(col(fc)).alias('max'),
                            spark_round(avg(col(fc)),2).alias('avg')).collect()[0]
        status = 'OK' if stats['min'] >= 0 else 'WARNING — negative values'
        print(f'  {fc:<22}: min={stats["min"]}  max={stats["max"]}  avg={stats["avg"]}  {status}')

print(f'\nTime: {round(time.time()-t0,2)}s')
print('Gold features check PASS')


**Stage 5: Batch inference** — load registered model, score all users, write to Gold.

In [0]:
# Load the registered model and score all 5.7M users
print('=== BATCH INFERENCE ===')

# Load model from registry
print(f'Loading model: {MODEL_NAME}')
try:
    model_uri = f'models:/{MODEL_NAME}/{MODEL_ALIAS}'
    model = mlflow.spark.load_model(model_uri)
    print(f'Model loaded: {model_uri}')
except Exception as e:
    print(f'Alias load failed: {e}')
    print('Trying latest version...')
    client = MlflowClient()
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")
    latest = sorted(versions, key=lambda v: int(v.version), reverse=True)[0]
    model_uri = f'models:/{MODEL_NAME}/{latest.version}'
    model = mlflow.spark.load_model(model_uri)
    print(f'Loaded version {latest.version}')

# Run inference
print()
print('Scoring all users...')
t0 = time.time()

feat = spark.table(GOLD_FEAT)
predictions = model.transform(feat)

# Extract probability and label
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType
prob_udf = udf(lambda v: float(v[1]), DoubleType())

final_preds = predictions.select(
    col('user_id'),
    prob_udf(col('probability')).alias('purchase_probability'),
    col('prediction').cast('int').alias('predicted_buyer'),
).withColumn('model_version', col('user_id') * 0 + 0)  # placeholder; real: lit(MODEL_ALIAS)

# Use lit() for string constant
from pyspark.sql.functions import lit
final_preds = final_preds.drop('model_version').withColumn('model_version', lit(MODEL_ALIAS))

# Write to Gold
(
    final_preds.writeTo(GOLD_PRED)
    .using('delta')
    .createOrReplace()
)

t_inf = round(time.time() - t0, 2)
n_pred = final_preds.count()
n_buyers = final_preds.filter(col('predicted_buyer') == 1).count()

print(f'Users scored    : {n_pred:,}')
print(f'Predicted buyers: {n_buyers:,}  ({round(n_buyers/n_pred*100,1)}%)')
print(f'Inference time  : {t_inf}s')
print('Inference COMPLETE — written to Gold')


**Stage 6: Validate output** — row count, probability range, model_version column.

In [0]:
# Validate the inference output table
print('=== INFERENCE OUTPUT VALIDATION ===')

preds = spark.table(GOLD_PRED)
n_preds = preds.count()
n_feat  = spark.table(GOLD_FEAT).count()

# Row count check
rc_ok = n_preds >= n_feat
print(f'Row count check : {n_preds:,} predictions >= {n_feat:,} users  ->  {"PASS" if rc_ok else "FAIL"}')

# Probability range check
prob_stats = preds.select(
    spark_min('purchase_probability').alias('min_prob'),
    spark_max('purchase_probability').alias('max_prob'),
    spark_round(avg('purchase_probability'),4).alias('avg_prob')
).collect()[0]
range_ok = 0.0 <= prob_stats['min_prob'] <= 1.0 and 0.0 <= prob_stats['max_prob'] <= 1.0
print(f'Probability range: min={prob_stats["min_prob"]}  max={prob_stats["max_prob"]}  avg={prob_stats["avg_prob"]}  ->  {"PASS" if range_ok else "FAIL"}')

# Null check
n_null_prob = preds.filter(col('purchase_probability').isNull()).count()
print(f'Null probabilities: {n_null_prob}  ->  {"PASS" if n_null_prob==0 else "FAIL"}')

# Prediction distribution
print()
print('Buyer distribution:')
preds.groupBy('predicted_buyer','model_version').count().show()

# Overall pipeline result
print()
all_pass = rc_ok and range_ok and n_null_prob == 0
print(f'INFERENCE VALIDATION: {"ALL CHECKS PASSED" if all_pass else "WARNING — see above"}')


### Task 2 — Save Final Model

Confirm the final model is registered and print the full production model card.

In [0]:
# Confirm the final model is saved and print a production model card
print('=== FINAL MODEL — PRODUCTION MODEL CARD ===')
print()

client = MlflowClient()

try:
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")
    versions_sorted = sorted(versions, key=lambda v: int(v.version), reverse=True)

    print(f'Registered model : {MODEL_NAME}')
    print(f'Total versions   : {len(versions_sorted)}')
    print()

    for v in versions_sorted:
        print(f'  Version {v.version}')
        print(f'    Status  : {v.status}')
        print(f'    Run ID  : {v.run_id}')
        print(f'    Created : {v.creation_timestamp}')

        # Get metrics from the run
        try:
            run = client.get_run(v.run_id)
            metrics = run.data.metrics
            params  = run.data.params
            print(f'    AUC     : {metrics.get("auc", metrics.get("areaUnderROC", "N/A"))}')
            print(f'    Recall  : {metrics.get("recall", "N/A")}')
            print(f'    Params  : numTrees={params.get("numTrees","N/A")}  maxDepth={params.get("maxDepth","N/A")}')
        except Exception as me:
            print(f'    Metrics : could not retrieve ({me})')
        print()

    print('Model saved and confirmed in Unity Catalog MLflow registry.')
    print(f'Production URI: models:/{MODEL_NAME}/{MODEL_ALIAS}')

except Exception as e:
    print(f'Registry check failed: {e}')
    print('Model may be saved under a different name — check Day 7 notebook')


### Production Checks

**Cell 9** — Data quality assertions  
**Cell 10** — Model health + drift monitor

In [0]:
# Production-style data quality assertions
# In a real job these would raise alerts / write to an audit table
print('=== DATA QUALITY ASSERTIONS ===')
print()

preds  = spark.table(GOLD_PRED)
feat   = spark.table(GOLD_FEAT)
silver = spark.table(SILVER_TBL)

checks = []

def check(name, condition, detail=''):
    status = 'PASS' if condition else 'FAIL'
    icon   = 'ok' if condition else 'WARN'
    checks.append({'name': name, 'status': status})
    print(f'  [{icon}] {name:<50} {status}  {detail}')

# Row count checks
n_preds  = preds.count()
n_feat   = feat.count()
n_silver = silver.count()
n_buyers = preds.filter(col('predicted_buyer') == 1).count()

check('Predictions row count >= feature store count',   n_preds >= n_feat,          f'{n_preds:,} >= {n_feat:,}')
check('Predicted buyers > 0',                          n_buyers > 0,               f'{n_buyers:,} buyers')
check('Buyer rate is plausible (5-30%)',                0.05 < n_buyers/n_preds < 0.30, f'{round(n_buyers/n_preds*100,1)}%')

# Probability checks
prob_min = preds.select(spark_min('purchase_probability')).collect()[0][0]
prob_max = preds.select(spark_max('purchase_probability')).collect()[0][0]
n_nulls  = preds.filter(col('purchase_probability').isNull()).count()

check('No null purchase_probability',                   n_nulls == 0,               f'{n_nulls} nulls')
check('Probabilities in valid range [0, 1]',            prob_min >= 0 and prob_max <= 1, f'min={prob_min:.4f}  max={prob_max:.4f}')

# Silver quality
n_null_user = silver.filter(col('user_id').isNull()).count() if 'user_id' in silver.columns else 0
check('No null user_id in Silver',                      n_null_user == 0,           f'{n_null_user} nulls')

print()
n_pass = sum(1 for c in checks if c['status'] == 'PASS')
print(f'Results: {n_pass}/{len(checks)} checks passed')
if n_pass < len(checks):
    print('WARNING: Some checks failed — review above before production deployment')
else:
    print('All data quality assertions PASSED')


In [0]:
# Model health check + distribution drift monitor
print('=== MODEL HEALTH + DRIFT MONITOR ===')
print()

# ── 1. Model health check from MLflow ────────────────────────────────
print('--- Model Health ---')
model_ok = False
try:
    client = MlflowClient()
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")
    latest = sorted(versions, key=lambda v: int(v.version), reverse=True)[0]
    run = client.get_run(latest.run_id)
    metrics = run.data.metrics

    auc    = metrics.get('auc', metrics.get('areaUnderROC', 0))
    recall = metrics.get('recall', 0)

    auc_ok    = auc >= MIN_AUC
    recall_ok = recall >= MIN_RECALL
    model_ok  = auc_ok and recall_ok

    print(f'  AUC    : {auc:.4f}  (threshold {MIN_AUC})   -> {"OK" if auc_ok else "BELOW THRESHOLD"}')
    print(f'  Recall : {recall:.4f} (threshold {MIN_RECALL})  -> {"OK" if recall_ok else "BELOW THRESHOLD"}')
    print(f'  Model health: {"PASS" if model_ok else "FAIL — consider retraining"}')
except Exception as e:
    print(f'  Could not check model metrics: {e}')
    print('  Verify model was logged with auc/recall metrics in Day 6')

print()

# ── 2. Drift monitor ─────────────────────────────────────────────────
print('--- Distribution Drift Monitor ---')
silver = spark.table(SILVER_TBL)
total  = silver.count()
buys   = silver.filter(col('event_type') == 'purchase').count()
current_rate = round(buys / total * 100, 2)
rel_change   = abs(current_rate - BASELINE_PURCHASE) / BASELINE_PURCHASE * 100
drift_ok     = rel_change <= DRIFT_THRESHOLD

print(f'  Baseline purchase rate : {BASELINE_PURCHASE}%')
print(f'  Current purchase rate  : {current_rate}%')
print(f'  Relative change        : {round(rel_change,1)}%  (threshold: {DRIFT_THRESHOLD}%)')
print(f'  Drift status           : {"NO DRIFT" if drift_ok else "DRIFT DETECTED — review retraining triggers"}')

print()

# ── 3. Final summary ─────────────────────────────────────────────────
print('=== FINAL PRODUCTION PIPELINE SUMMARY ===')
print(f'  Bronze rows     : {total:,}')
print(f'  Silver rows     : {total:,}')
print(f'  User features   : {spark.table(GOLD_FEAT).count():,}')
print(f'  Predictions     : {spark.table(GOLD_PRED).count():,}')
print(f'  Model version   : {MODEL_ALIAS}')
print(f'  Drift check     : {"PASS" if drift_ok else "ACTION REQUIRED"}')
print()
print('14-Day Databricks AI Challenge — COMPLETE')


## Day 14 — Challenge Complete!

| Cell | Task | What it does |
|------|------|--------------|
| 2 | **Task 1** | Pipeline health check — all table row counts |
| 3 | Task 1 | Bronze validation — schema + row count assertion |
| 4 | Task 1 | Silver quality check — null rates + partition confirmation |
| 5 | Task 1 | Gold features check — shape + feature ranges |
| 6 | Task 1 | **Full batch inference** — load model → score 5.7M → write Gold |
| 7 | Task 1 | Output validation — row count + probability range checks |
| 8 | **Task 2** | Save + confirm final model — full production model card |
| 9 | Production | Data quality assertions — 6 automated checks |
| 10 | Production | Model health + drift monitor — AUC, recall, purchase rate |

---

### 14-Day Challenge Complete

**What we built:**
- 109M events → Bronze → Silver → Gold (5.7M user features)
- Random Forest model: AUC 0.9657 · Recall 92.6% · ~799K predicted buyers
- ALS recommendations: Top-5 products per user
- Full production pipeline with monitoring and failure handling

**Hosted by:** Indian Data Club — [discord.gg/wpWQgM49](https://discord.gg/wpWQgM49)